# Workflow 1: Flat React Agent

In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
from typing import List, Dict, Any

import uuid
import operator
from typing import TypedDict, Literal, Optional, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

from configs.setting import settings
from configs.GetConfig import config

from src.LLMService import LLMService
from src.e_agents.guardrail_call import GuardrailCall
from src.e_agents.rejection_call import RejectionCall

from src.d_tools import (
    product_search, 
    product_compare,
    check_stock,
    policy_search,
    order_lookup,
)

from src.d_tools import (
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    CHECK_STOCK_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
)

from src.f_prompts import (
    FULL_MASTER_PROMPT,        
    FULL_REJECTION_PROMPT
)

from app.core.security import verify_supabase_jwt

In [2]:
available_tools = {
    "product_search": product_search,
    "product_compare": product_compare,
    "check_stock": check_stock,
    "policy_search": policy_search,
    "order_lookup": order_lookup
}

tools_schema = [
    PRODUCT_SEARCH_SCHEMA,
    PRODUCT_COMPARE_SCHEMA,
    CHECK_STOCK_SCHEMA,
    POLICY_SEARCH_SCHEMA,
    ORDER_LOOKUP_SCHEMA
]

In [3]:
class MasterAgent:
    def __init__(self, llm_service: LLMService, config):
        self.llm_service = llm_service
        self.model = config.llm.google.available[0]  

    def invoke(
        self, 
        messages: List[Dict[str, Any]],
        available_tools: Dict[str, Any] = None,
        tools_schema: List[Dict[str, Any]] = None
        ):

        start_time = time.time()
        first_token_time = None
        total_input_tokens = 0
        total_output_tokens = 0
        
        max_turns = config.agent.max_turns
        for turn in range(max_turns):
            turn_start_time = time.time()
            first_token_time = None
            turn_input_tokens = 0
            turn_output_tokens = 0

            print(f"🌀 --- LƯỢT {turn + 1} (STREAMING) ---")
            
            # Gọi API Gemini với streaming
            response = self.llm_service.call_gemini(
                model=self.model,
                messages=messages,
                tools=tools_schema
            )
            
            text_content = ""
            tool_calls_dict = {}
            
            for chunk in response:
                if getattr(chunk, "usage_metadata", None):
                    turn_input_tokens = chunk.usage_metadata.prompt_token_count or 0
                    turn_output_tokens = chunk.usage_metadata.candidates_token_count or 0

                chunk_text = next((p.text for cand in (chunk.candidates or []) for p in (cand.content.parts or []) if getattr(p, "text", None)), "")
                if first_token_time is None and (chunk_text or chunk.function_calls):
                    first_token_time = time.time() - turn_start_time
                    print(f"⏱️ [TTFT] Chữ/Tool đầu tiên xuất hiện sau: {first_token_time:.2f}s\n")

                if chunk_text:
                    text_content += chunk_text
                
                if chunk.function_calls:
                    for idx, call in enumerate(chunk.function_calls):
                        sig = None
                        if chunk.candidates and chunk.candidates[0].content and chunk.candidates[0].content.parts:
                            for p in chunk.candidates[0].content.parts:
                                if getattr(p, 'function_call', None) and getattr(p, 'thought_signature', None):
                                    sig = p.thought_signature

                        tool_calls_dict[idx] = {
                            "id": f"call_gemini_{turn}_{idx}",
                            "name": call.name,
                            "arguments": json.dumps(call.args) if isinstance(call.args, dict) else str(call.args),
                            "thought_signature": sig
                        }

            turn_elapsed = time.time() - turn_start_time
            total_input_tokens += turn_input_tokens
            total_output_tokens += turn_output_tokens
            
            print(f"\n\n⏱️ [TURN {turn + 1} LATENCY]: {turn_elapsed:.2f}s")
            print(f"📊 [TURN {turn + 1} TOKENS]: Input = {turn_input_tokens} | Output = {turn_output_tokens} | Subtotal = {turn_input_tokens + turn_output_tokens}")

            # Format lại tool_calls thành cấu trúc chuẩn OpenAI để lưu history
            formatted_tool_calls = []
            for idx, tc in tool_calls_dict.items():
                item = {
                    "id": tc["id"],
                    "type": "function",
                    "function": {
                        "name": tc["name"],
                        "arguments": tc["arguments"]
                    }
                }
                if tc.get("thought_signature"):
                    item["thought_signature"] = tc["thought_signature"]
                formatted_tool_calls.append(item)

            # Lưu message của assistant vào history
            agent_msg = {
                "role": "assistant",
                "content": text_content if text_content else None
            }
            if formatted_tool_calls:
                agent_msg["tool_calls"] = formatted_tool_calls
                
            messages.append(agent_msg)
            
            # Thực thi Tools nếu LLM yêu cầu
            if formatted_tool_calls:
                print("\n🔧 LLM yêu cầu gọi Tool...")
                tool_start_time = time.time()
                
                for tc in formatted_tool_calls:
                    func_name = tc["function"]["name"]
                    func_args = json.loads(tc["function"]["arguments"]) if tc["function"]["arguments"] else {}
                    
                    if func_name in available_tools:
                        real_function = available_tools[func_name]
                        print(f"   👉 Chạy hàm: {func_name}({func_args})")
                        result = real_function(**func_args)
                        print(f"   📊 Kết quả từ Tool: {result}")
                        
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tc["id"],
                            "name": func_name,
                            "content": str(result)
                        })
                    else:
                        print(f"   ❌ Lỗi: Không tìm thấy tool '{func_name}' trong available_tools!")
                
                tool_elapsed = time.time() - tool_start_time
                print(f"⏱️ [TOOL EXECUTION TIME]: {tool_elapsed:.2f}s")
                print("🔄 Gửi kết quả Tool lại cho Gemini suy luận tiếp...\n")
                
                # Nghỉ 1s trước khi sang lượt mới
                time.sleep(1)
                continue
            else:
                total_elapsed = time.time() - start_time
                print("\n==================================================")
                print(f"✅ --- HOÀN THÀNH HOÀN TOÀN ---")
                print(f"⏱️ [TOTAL AGENT LATENCY]: {total_elapsed:.2f}s")
                print(f"📊 [TOTAL AGENT TOKENS]: Input = {total_input_tokens} | Output = {total_output_tokens} | Grand Total = {total_input_tokens + total_output_tokens}")
                print(f"TPM: {(total_input_tokens + total_output_tokens) * 60 / total_elapsed}")
                print("==================================================\n")
                return {
                    "content": text_content if text_content else None,
                    "latency": total_elapsed,
                    "tokens": {
                        "input": total_input_tokens,
                        "output": total_output_tokens,
                    }
                }



In [4]:
llm_service = LLMService(settings, config)
guardrail_call = GuardrailCall(llm_service, config)
rejection_call = RejectionCall(llm_service, config)
master_agent = MasterAgent(llm_service, config)

In [5]:
class RetrievedChunk(TypedDict):
    content: str
    source: str
    score: float
    chunk_type: str

class AgentState(TypedDict):

    # 1. INput user
    user_query: str
    session_id: str

    # 2. Auth
    user_token: Optional[str]
    user_id: Optional[str]
    is_authenticated: bool

    # 3. Guardrail & Quality
    risk_level: Optional[str]               # "low" | "medium" | "high"
    relevance_score: float                   # Dùng cho self-check Corrective RAG

    # 4. Router (Định tuyến)
    intent: str                              # Kết quả phân loại: 'product', 'policy', 'account', 'support'
    selected_agent: Optional[str]            # Quyết định Nút xử lý tiếp theo

    # 5. Retrieval & Tools
    retrieved_context: list[RetrievedChunk]
    tool_calls_used: Annotated[list[str], operator.add]  # ⚡ Reducer cộng dồn tool đã gọi
    iteration_count: int                     # Đếm số lần lặp chống infinite loop

    # 6. Hội thoại
    messages: Annotated[list, add_messages]  # ⚡ Reducer cộng dồn tin nhắn
    conversation_state: dict

    # 7. Output
    final_answer: Optional[str]
    cited_sources: list[str]
    ticket_id: Optional[str]
    show_popup: bool

    # 8. Thống kê
    input_tokens: Annotated[int, operator.add]
    output_tokens: Annotated[int, operator.add]
    latency: Annotated[float, operator.add]
    total_tokens: Annotated[int, operator.add]

In [6]:
def receive_node(state: AgentState) -> dict:
    query = state.get("user_query", "").strip()
    token = state.get("user_token")
    user_id = None

    if token:
        user_id = verify_supabase_jwt(token)
    
    is_authenticated = True if user_id else False
    if is_authenticated:
        print(f"Người dùng đã xác thực")
    else:
        print(f"Người dùng chưa xác thực")

    return {
        "user_id": user_id,
        "is_authenticated": is_authenticated,
        "user_query": query  
    }

In [7]:
def guardrail_node(state: AgentState) -> dict:
    query = state["user_query"]
    
    result = guardrail_call.invoke(query)
    risk_level = result["risk_level"]
    
    # Chỉ lưu tin nhắn khi KHÔNG phải attack
    if risk_level != "attack":
        return {
            "risk_level": risk_level,
            "show_popup": False,
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ],
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }
    else:
        # Attack → KHÔNG lưu
        return {
            "risk_level": risk_level,
            "show_popup": True,
            "latency": result["latency"],
            "input_tokens": result["tokens"]["input"],
            "output_tokens": result["tokens"]["output"],
            "total_tokens": result["tokens"]["input"] + result["tokens"]["output"]
        }

In [8]:
def rejection_node(state: AgentState) -> dict:
    """
    [NODE] Rejection Agent (Từ chối):
    - Chỉ chạy khi risk_level == "needs_ticket"
    - Trả về câu từ chối lịch sự
    - KHÔNG lưu tin nhắn vào messages (đã lưu ở guardrail_node)
    """
    
    query = state["user_query"]

    result = rejection_call.invoke(query)

    return {
        "final_answer": result["content"],
        "show_popup": True,
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [9]:
def master_node(state: AgentState) -> dict:
    """
    [NODE 3] Master Agent (Tư duy):
    - Chỉ chạy khi risk_level != "attack"
    - Build messages với system prompt + lịch sử chat
    - MasterAgent.invoke KHÔNG tự động lưu tin nhắn, phải tự append
    - Chỉ lưu assistant message vào state["messages"] (KHÔNG lưu system prompt)
    """   
    # Build messages cho master agent
    full_messages = [
        {
            "role": "system",
            "content": FULL_MASTER_PROMPT
        }
    ] + state["messages"]
    
    # Gọi master agent
    result = master_agent.invoke(
        messages=full_messages,
        available_tools=available_tools,
        tools_schema=tools_schema
    )
    
    # Append assistant message vào state["messages"] (KHÔNG có system prompt)
    assistant_msg = {
        "role": "assistant",
        "content": result["content"]
    }
    
    return {
        "final_answer": result["content"],
        "messages": [assistant_msg],
        "input_tokens": result["tokens"]["input"],
        "output_tokens": result["tokens"]["output"],
        "total_tokens": result["tokens"]["input"] + result["tokens"]["output"],
        "latency": result["latency"]
    }

In [10]:
builder = StateGraph(AgentState)

builder.add_node("receive_node", receive_node)
builder.add_node("guardrail_node", guardrail_node)
builder.add_node("rejection_node", rejection_node)
builder.add_node("master_node", master_node)

def route_after_guardrail(state: AgentState) -> str:
    """Hàm quyết định Nút tiếp theo dựa vào kết quả của Guardrail"""
    risk = state.get("risk_level", "safe")
    
    if risk == "attack":
        print("⚠️ [ROUTER] Phát hiện ATTACK ➡️ Rẽ nhánh sang rejection_node")
        return "rejection_node"
    else:
        print("✅ [ROUTER] An toàn SAFE ➡️ Rẽ nhánh sang master_node")
        return "master_node"


builder.add_edge(START, "receive_node")
builder.add_edge("receive_node", "guardrail_node")
builder.add_conditional_edges(
    "guardrail_node",
    route_after_guardrail,
    {
        "rejection_node": "rejection_node", 
        "master_node": "master_node"       
    }
)
builder.add_edge("rejection_node", END)
builder.add_edge("master_node", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)
print("🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1!")

🎉 Đã dựng thành công Đồ thị LangGraph Workflow 1!


In [11]:
run_config = {"configurable": {"thread_id": "session_test_notebook_001"}}

# 1. Gọi Đồ thị chạy
res = app.invoke(
    {"user_query": "ok, vậy macbook air M5 giá bao nhiêu ạ, và còn hàng không ạ?"}, 
    config=run_config
)

# 2. IN BÁO CÁO THỐNG KÊ CHI TIẾT TỪ AGENT STATE
print("="*60)
print("📊 BÁO CÁO THỐNG KÊ CHI TIẾT (AGENT STATE METRICS)")
print("="*60)
print(f"💬 Câu trả lời (Final Answer) : {res.get('final_answer')}")
print(f"🛡️ Mức độ rủi ro (Risk Level) : {res.get('risk_level')}")
print(f"⏱️ Tổng độ trễ (Total Latency): {res.get('latency', 0):.2f}s")
print(f"📥 Input Tokens               : {res.get('input_tokens', 0)}")
print(f"📤 Output Tokens              : {res.get('output_tokens', 0)}")
print(f"🧮 Tổng Tokens (Total Tokens)  : {res.get('total_tokens', 0)}")

# 3. IN LỊCH SỬ HỘI THOẠI TRONG MEMORY
print("\n📜 LỊCH SỬ HỘI THOẠI (MESSAGES HISTORY):")
for idx, msg in enumerate(res.get("messages", []), 1):
    role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else "unknown")
    content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else "")
    print(f"  [{idx}] {role.upper()}: {content}")

print("="*60)


Người dùng chưa xác thực
✅ [ROUTER] An toàn SAFE ➡️ Rẽ nhánh sang master_node
🌀 --- LƯỢT 1 (STREAMING) ---
⏱️ [TTFT] Chữ/Tool đầu tiên xuất hiện sau: 0.62s



⏱️ [TURN 1 LATENCY]: 0.62s
📊 [TURN 1 TOKENS]: Input = 3301 | Output = 22 | Subtotal = 3323

🔧 LLM yêu cầu gọi Tool...
   👉 Chạy hàm: product_search({'key_word': 'Macbook Air M5'})
   📊 Kết quả từ Tool: Product MacBook Air M5 13 inch 2026 10CPU 10GPU 16GB 1TB | Chính hãng Apple Việt Nam
- Brand: Apple
- Price: 43,690,000 VNĐ
- Score: 0.8396
- Details:
Sản phẩm: MacBook Air M5 13 inch 2026 10CPU 10GPU 16GB 1TB | Chính hãng Apple Việt Nam

Thương hiệu: Apple | Danh mục: laptop

Thông tin giá & Kho hàng:
- Giá thực tế: 43,690,000 VNĐ
- Giá gốc niêm yết: 44,250,000 VNĐ
- Mức giảm giá: 1.27%

Thông số kỹ thuật chi tiết:
- Bộ vi xử lý (CPU/Chipset): Chip Apple M5
CPU 10 lõi với 4 lõi siêu xử lý và 6 lõi tiết kiệm điện
- Dung lượng RAM: 16GB
- Dung lượng lưu trữ: 1TB
- Kích thước màn hình: 13.6 inches
- Độ phân giải màn hình: 2560 x 1664